# EchoGesture - GPU training on Colab

**Before running:** Runtime -> Change runtime type -> **T4 GPU**.

This trains only the classifier on cached landmarks. Extraction, INT8 export and evaluation stay on your laptop.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch; print('CUDA available:', torch.cuda.is_available())

## 1. Upload `colab_bundle.zip`
(made on your laptop with `python make_colab_bundle.py`)

In [ ]:
from google.colab import files
import os, zipfile
up = files.upload()
name = next(iter(up))
!rm -rf /content/echo && mkdir -p /content/echo
zipfile.ZipFile(name).extractall('/content/echo')
!ls /content/echo/data/landmarks | wc -l

## 2e. Model F: up to 30 ASL Citizen clips per word + pixel augmentation - RUN THIS ONE (22 Nov)
Needs the new `colab_bundle.zip` (about 41k ASL Citizen clips). Downloads `echo_d256_aslc30_pix.pt` when done. **Cutoff: 15:00** - if it is not done by then, we ship E.


In [ ]:
%cd /content/echo/src
!wc -l ../data/aslc_index.csv && ls ../data/landmarks | grep -c aslc_ && ls ../data/landmarks_pix | wc -l
!python scripts/train.py --epochs 100 --final --workers 2 --bs 128 --dim 256 --lr 3e-3 --out models/echo_d256_aslc30_pix.pt
from google.colab import files
files.download('/content/echo/src/models/echo_d256_aslc30_pix.pt')

## 2d. Model E: D + pixel augmentation - already trained, 57.0% - skip
The same as D plus 7,826 compressed/over-exposed/colour-shifted/occluded copies (from `data/landmarks_pix`, picked up automatically). It downloads `echo_d256_aslc_pix.pt` when done.


In [ ]:
%cd /content/echo/src
!ls ../data/landmarks_pix | wc -l
!python scripts/train.py --epochs 100 --final --workers 2 --bs 128 --dim 256 --lr 3e-3 --out models/echo_d256_aslc_pix.pt
from google.colab import files
files.download('/content/echo/src/models/echo_d256_aslc_pix.pt')

## 2c. Model D: WLASL + ASL Citizen (already trained, 56.2%) - skip


In [ ]:
%cd /content/echo/src
!ls ../data/aslc_index.csv && ls ../data/landmarks | grep -c aslc_
!python scripts/train.py --epochs 100 --final --workers 2 --bs 128 --dim 256 --lr 3e-3 --out models/echo_d256_aslc.pt
from google.colab import files
files.download('/content/echo/src/models/echo_d256_aslc.pt')

## 2b. Model C (second seed) - optional, skip unless you have spare GPU time


In [ ]:
%cd /content/echo/src
!python scripts/train.py --epochs 100 --final --workers 2 --bs 128 --dim 256 --lr 3e-3 --seed 1 --out models/echo_d256_s1.pt
from google.colab import files
files.download('/content/echo/src/models/echo_d256_s1.pt')

## 2. Model A (already trained - skip)


In [ ]:
%cd /content/echo/src
!python scripts/train.py --epochs 100 --final --workers 2 --bs 128 --dim 256 --lr 3e-3 --out models/echo_d256.pt
from google.colab import files
files.download('/content/echo/src/models/echo_d256.pt')

## 3. Model B (large - already trained, worse than A - skip)


In [ ]:
%cd /content/echo/src
!python scripts/train.py --epochs 100 --final --workers 2 --bs 128 --dim 384 --lr 2e-3 --drop 0.25 --out models/echo_d384.pt
from google.colab import files
files.download('/content/echo/src/models/echo_d384.pt')

## If it disconnects mid-training
The last saved epoch is in `/content/echo/src/models/` - download it from the Files panel (left sidebar) before the runtime resets.